Task 2 Lineage-Holdout Significance Evaluation

Tests whether the lineage-holdout improvement in Precision@10/Recall@10 for WHO-catalogued variant recovery is statistically significant against the random split baseline, per architecture, and applies a Holm-Bonferroni correction across the 4 architectures within each metric (precision, recall).

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from pathlib import Path

LINEAGE_DIR = "../results/test_output_summary/task2/lineage_split"
RANDOM_DIR = "../results/test_output_summary/task2/random_split"

ARCH_FILES = {
    "SD-LogReg": ("all_lineage_summary_logreg.csv", "random_split_summmary_logreg.csv"),
    "SD-CNN": ("all_lineage_summary_sdcnn.csv", "random_split_summary_sdcnn.csv"),
    "Evo2-CNN-zero-shot": ("all_lineage_summary_evo2_zeroshot_token.csv", "random_split_summary_evo2_zeroshot_token.csv"),
    "Evo2-CNN-pca10": ("all_lineage_summary_evo2_zeroshot_pca.csv", "random_split_summary_evo2_zeroshot_pca.csv"),
}

OUT_FILE = "../results/significance_testing/table21_lineage_vs_random_significance_holm.csv"

In [6]:
def load_and_merge_k10(arch_name, lineage_file, random_file):
    """Load lineage and random split CSVs for k=10, compute deltas per drug."""
    lineage_df = pd.read_csv(Path(LINEAGE_DIR) / lineage_file)
    random_df = pd.read_csv(Path(RANDOM_DIR) / random_file)
    
    lineage_k10 = lineage_df[lineage_df['k'] == 10][['drug', 'mean_P@k', 'mean_R@k']].copy()
    random_k10 = random_df[random_df['k'] == 10][['drug', 'P@k', 'R@k']].copy()
    
    merged = lineage_k10.merge(random_k10, on='drug', suffixes=('_lineage', '_random'))
    merged['delta_p10'] = merged['mean_P@k'] - merged['P@k']
    merged['delta_r10'] = merged['mean_R@k'] - merged['R@k']
    merged['architecture'] = arch_name
    
    return merged[['architecture', 'drug', 'mean_P@k', 'P@k', 'delta_p10', 'mean_R@k', 'R@k', 'delta_r10']]

rows_raw = []
for arch, (lineage_file, random_file) in ARCH_FILES.items():
    merged = load_and_merge_k10(arch, lineage_file, random_file)
    rows_raw.append(merged)

df = pd.concat(rows_raw, ignore_index=True)

rows = []
for arch in sorted(df["architecture"].unique()):
    sub = df[df["architecture"] == arch].sort_values("drug")
    _, p_precision10 = wilcoxon(sub["mean_P@k"], sub["P@k"], alternative="two-sided")
    _, p_recall10 = wilcoxon(sub["mean_R@k"], sub["R@k"], alternative="two-sided")
    rows.append({
        "architecture": arch,
        "mean_delta_p10": sub["delta_p10"].mean(),
        "mean_delta_r10": sub["delta_r10"].mean(),
        "p_precision10": p_precision10,
        "p_recall10": p_recall10,
        "n_drugs": len(sub),
    })

result = pd.DataFrame(rows)

result["p_precision10_holm"] = multipletests(result["p_precision10"], method="holm")[1]
result["p_recall10_holm"] = multipletests(result["p_recall10"], method="holm")[1]

result.to_csv(OUT_FILE, index=False)
print(f"[OK] Wrote {OUT_FILE} with {len(result)} rows.")
result

[OK] Wrote ../results/significance_testing/task2_lineage_significance_holm.csv with 4 rows.


/home/saishradhamo_umass_edu/.local/lib/python3.8/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
/home/saishradhamo_umass_edu/.local/lib/python3.8/site-packages/scipy/stats/_morestats.py:3428: UserWarning: Sample size too small for normal approximation.
  warnings.warn("Sample size too small for normal approximation.")


,architecture,mean_delta_p10,mean_delta_r10,p_precision10,p_recall10,n_drugs,p_precision10_holm,p_recall10_holm
0,Evo2-CNN-pca10,-0.0375,0.013929,1.000000,0.674424,10,1.000000,1.000000
1,Evo2-CNN-zero-shot,-0.1175,-0.075558,0.017148,0.007686,10,0.068592,0.030743
2,SD-CNN,0.0300,-0.021942,0.511692,0.275391,10,1.000000,0.826172
3,SD-LogReg,0.0175,0.038509,0.593631,0.514670,10,1.000000,1.000000
